#### 08 — Embedding-Based Classification

##### Purpose

In the previous NLP baseline, support-ticket text was represented using TF-IDF:

```  text

Ticket text
    ↓
TF-IDF
    ↓
85 vocabulary-based features
    ↓
Logistic Regression
    ↓
Ticket category

```

In this notebook, we replace TF-IDF with pretrained sentence embeddings:

``` text

Ticket text
    ↓
Sentence Embedding Model
    ↓
384 semantic features
    ↓
Logistic Regression
    ↓
Ticket category

```

The classifier remains Logistic Regression.

This creates a controlled experiment:
    
``` text

Experiment A

TF-IDF
  ↓
Logistic Regression


Experiment B

Embeddings
  ↓
Logistic Regression

```
Therefore, if performance changes, the main change is the text representation, not the classifier.


##### Technologies

``` text

Unity Catalog / Delta
        ↓
Persisted modeling dataset

PySpark
        ↓
Load existing train / validation / test splits

Sentence Transformers
        ↓
Pretrained semantic text embeddings

NumPy
        ↓
Dense embedding matrices

scikit-learn
        ↓
Logistic Regression
Evaluation metrics
Cosine similarity

```


##### Input

We continue using the persisted modeling table: dbw_agentic_ai_dev.support_ticket_ai.nlp_modeling_dataset

with the existing split assignments:

- train       70
- validation  17
- test        13

We do not create new train/test splits.


##### Output

By the end of the notebook we will have:

- X_train_embeddings
- X_validation_embeddings
- X_test_embeddings
- embedding_classifier
- validation predictions
- test predictions
- accuracy
- macro F1
- weighted F1
- classification report
- confusion matrix
- prediction-level results

And, most importantly, we'll compare the embedding model against our previous TF-IDF baseline.


##### Architecture:

``` text


                     ┌─────────────────────────┐
                     │ nlp_modeling_dataset    │
                     └────────────┬────────────┘
                                  │
                    Existing persisted splits
                                  │
                 ┌────────────────┼────────────────┐
                 ↓                ↓                ↓
               Train         Validation          Test
                 │                │                │
                 └────────────────┼────────────────┘
                                  ↓
                    SentenceTransformer
                                  ↓
                      Dense Embeddings
                                  ↓
                 ┌────────────────┼────────────────┐
                 ↓                ↓                ↓
             (70, 384)        (17, 384)        (13, 384)
                 │
                 ↓
          Logistic Regression
                 │
                 ↓
        Category Predictions
                 │
                 ↓
             Evaluation

```

##### 1 - Install the Embedding Library

In [0]:
%pip install sentence-transformers

##### 2 - Imports

In [0]:
import numpy as np
import pandas as pd

from sentence_transformers import (
    SentenceTransformer,
)

from sklearn.metrics.pairwise import (
    cosine_similarity,
)

from src.project_config import (
    CLEAN_TEXT_COL,
    TARGET_COL,
    TICKET_ID_COL,
)

from src.data_preparation import (
    load_modeling_dataset,
    validate_modeling_schema,
    split_modeling_dataset,
    to_pandas_modeling_data,
)

from src.model_training import (
    train_logistic_regression,
)

from src.model_evaluation import (
    calculate_classification_metrics,
    build_classification_report,
    build_confusion_matrix,
    build_prediction_results,
)

from src.feature_engineering import (
    load_embedding_model,
    generate_embeddings,
    build_embedding_features,
)

##### 3 - Load the Persisted Modeling Dataset

In [0]:
modeling_sdf = load_modeling_dataset(
    spark
)

validate_modeling_schema(
    modeling_sdf
)

In [0]:
display(
    modeling_sdf
)

In [0]:
modeling_sdf.printSchema()

##### 4 - Retrieve the Existing Splits

In [0]:
(
    train_sdf,
    validation_sdf,
    test_sdf,
) = split_modeling_dataset(
    modeling_sdf
)

In [0]:
train_count = train_sdf.count()
validation_count = validation_sdf.count()
test_count = test_sdf.count()

print(
    f"""
Training rows   : {train_count}
Validation rows : {validation_count}
Test rows       : {test_count}
"""
)

##### 5 - Convert Modeling Data to Pandas

In [0]:
train_pdf = to_pandas_modeling_data(
    train_sdf
)

validation_pdf = to_pandas_modeling_data(
    validation_sdf
)

test_pdf = to_pandas_modeling_data(
    test_sdf
)

In [0]:
display(
    train_pdf.head()
)

In [0]:
X_train_text = (
    train_pdf[
        CLEAN_TEXT_COL
    ]
    .fillna("")
    .astype(str)
)

X_validation_text = (
    validation_pdf[
        CLEAN_TEXT_COL
    ]
    .fillna("")
    .astype(str)
)

X_test_text = (
    test_pdf[
        CLEAN_TEXT_COL
    ]
    .fillna("")
    .astype(str)
)

In [0]:
y_train = train_pdf[
    TARGET_COL
]

y_validation = validation_pdf[
    TARGET_COL
]

y_test = test_pdf[
    TARGET_COL
]

##### 6 - Choose the Sentence Embedding Model

In [0]:
#This is a pretrained sentence embedding model.
EMBEDDING_MODEL_NAME = ("sentence-transformers/all-MiniLM-L6-v2")

In [0]:
embedding_model = (
    load_embedding_model()
)

In [0]:
embedding_dimension = (
    embedding_model
    .get_sentence_embedding_dimension()
)

print(
    "Embedding dimension:",
    embedding_dimension,
)

In [0]:
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

In [0]:
embedding_dimension = (
    embedding_model
    .get_sentence_embedding_dimension()
)

print(
    "Embedding model:",
    EMBEDDING_MODEL_NAME,
)

print(
    "Embedding dimension:",
    embedding_dimension,
)

##### 7 : Understand encode() Before Hiding It in src

In [0]:
sample_text = (
    "my router keeps disconnecting"
)

sample_embedding = (
    embedding_model.encode(
        sample_text
    )
)

In [0]:
print(
    "Embedding type:",
    type(sample_embedding),
)

print(
    "Embedding shape:",
    sample_embedding.shape,
)

In [0]:
print(
    sample_embedding[:10]
)

##### 8 - Explore Semantic Similarity

In [0]:
technical_ticket_1 = (
    "my router keeps disconnecting"
)

technical_ticket_2 = (
    "my internet connection keeps dropping"
)

billing_ticket = (
    "i was charged twice on my monthly bill"
)

In [0]:
example_texts = [
    technical_ticket_1,
    technical_ticket_2,
    billing_ticket,
]

example_embeddings = (
    embedding_model.encode(
        example_texts
    )
)

In [0]:
print(
    "Example embedding shape:",
    example_embeddings.shape,
)

In [0]:
example_embeddings

##### 9 - Calculate Cosine Similarity

In [0]:
technical_similarity = (
    cosine_similarity(
        example_embeddings[0:1],
        example_embeddings[1:2],
    )[0][0]
)

billing_similarity = (
    cosine_similarity(
        example_embeddings[0:1],
        example_embeddings[2:3],
    )[0][0]
)

In [0]:
print(
    f"""
Router vs Internet similarity:
{technical_similarity:.4f}

Router vs Billing similarity:
{billing_similarity:.4f}
"""
)

##### 10 - Generate Embeddings for the Actual Dataset

In [0]:
(
    X_train_embeddings,
    X_validation_embeddings,
    X_test_embeddings,
) = build_embedding_features(
    embedding_model,
    X_train_text,
    X_validation_text,
    X_test_text,
)

In [0]:
print(
    f"""
Train embedding shape      :
{X_train_embeddings.shape}

Validation embedding shape :
{X_validation_embeddings.shape}

Test embedding shape       :
{X_test_embeddings.shape}
"""
)

##### 11 — Verify the Matrices

In [0]:
assert (
    X_train_embeddings.shape[0]
    == train_count
)

assert (
    X_validation_embeddings.shape[0]
    == validation_count
)

assert (
    X_test_embeddings.shape[0]
    == test_count
)

assert (
    X_train_embeddings.shape[1]
    == embedding_dimension
)

assert (
    X_validation_embeddings.shape[1]
    == embedding_dimension
)

assert (
    X_test_embeddings.shape[1]
    == embedding_dimension
)

print(
    "Embedding feature validation passed."
)

##### 15 — Train Logistic Regression

In [0]:
embedding_classifier = (
    train_logistic_regression(
        X_train_embeddings,
        y_train,
    )
)

16 — Inspect the Trained Classifier

In [0]:
print(
    "Classes:",
    embedding_classifier.classes_,
)

print(
    "Coefficient shape:",
    embedding_classifier.coef_.shape,
)

print(
    "Intercept shape:",
    embedding_classifier.intercept_.shape,
)

print(
    "Iterations:",
    embedding_classifier.n_iter_,
)

##### 17 — Validation Predictions

In [0]:
validation_predictions = (
    embedding_classifier.predict(
        X_validation_embeddings
    )
)

In [0]:
validation_metrics = (
    calculate_classification_metrics(
        y_validation,
        validation_predictions,
    )
)

In [0]:
print(
    f"""
Validation Accuracy   :
{validation_metrics['accuracy']:.4f}

Validation Macro F1   :
{validation_metrics['macro_f1']:.4f}

Validation Weighted F1:
{validation_metrics['weighted_f1']:.4f}
"""
)

##### 18 - Test Predictions

In [0]:
test_predictions = (
    embedding_classifier.predict(
        X_test_embeddings
    )
)

In [0]:
test_probabilities = (
    embedding_classifier.predict_proba(
        X_test_embeddings
    )
)

In [0]:
test_metrics = (
    calculate_classification_metrics(
        y_test,
        test_predictions,
    )
)

In [0]:
print(
    f"""
Test Accuracy   :
{test_metrics['accuracy']:.4f}

Test Macro F1   :
{test_metrics['macro_f1']:.4f}

Test Weighted F1:
{test_metrics['weighted_f1']:.4f}
"""
)

##### 19 — Classification Report

In [0]:
classification_report_df = (
    build_classification_report(
        y_test,
        test_predictions,
    )
)

display(
    classification_report_df
)

##### 20 — Confusion Matrix

In [0]:
class_labels = list(
    embedding_classifier.classes_
)

In [0]:
confusion_df = (
    build_confusion_matrix(
        y_test,
        test_predictions,
        class_labels,
    )
)

display(
    confusion_df
)

##### 21 — Prediction-Level Results

In [0]:
test_results_df = (
    build_prediction_results(
        ticket_ids=test_pdf[
            TICKET_ID_COL
        ],
        text=test_pdf[
            CLEAN_TEXT_COL
        ],
        y_true=y_test,
        y_pred=test_predictions,
        probabilities=test_probabilities,
        class_labels=class_labels,
    )
)

In [0]:
display(
    test_results_df
)

##### 22 — Inspect Errors

In [0]:
test_errors_df = (
    test_results_df[
        ~test_results_df[
            "is_correct"
        ]
    ]
    .copy()
)



In [0]:
test_errors_df

In [0]:
display(
    test_errors_df
)

In [0]:
test_error_count = len(
    test_errors_df
)

print(
    "Test errors:",
    test_error_count,
)

##### 23 — Specifically Inspect the Previous TF-IDF Failure

In [0]:
previous_failure_df = (
    test_results_df[
        test_results_df[
            CLEAN_TEXT_COL
        ]
        .str.contains(
            "account locked",
            case=False,
            na=False,
        )
    ]
)

display(
    previous_failure_df
)

##### 24 — Compare TF-IDF Baseline With Embeddings

In [0]:
tfidf_baseline = {
    "representation": "TF-IDF",
    "accuracy": 0.8462,
    "macro_f1": 0.7083,
    "weighted_f1": 0.7821,
    "test_errors": 2,
}

In [0]:
embedding_result = {
    "representation": "Embeddings",
    "accuracy": test_metrics[
        "accuracy"
    ],
    "macro_f1": test_metrics[
        "macro_f1"
    ],
    "weighted_f1": test_metrics[
        "weighted_f1"
    ],
    "test_errors": test_error_count,
}

In [0]:
comparison_df = pd.DataFrame(
    [
        tfidf_baseline,
        embedding_result,
    ]
)

In [0]:
display(
    comparison_df
)

##### 25 — Key Concept: Are Embeddings Automatically Better?

No. This is important.

We should not expect: Embeddings > TF-IDF for every dataset.

Our dataset is tiny:

- 100 tickets total
- 70 training tickets

and its categories contain fairly obvious keywords.

TF-IDF can perform extremely well on such datasets.

For example:

- refund
- invoice
- charged

are strong Billing clues.

- password
- login
- authentication

are strong Login clues.

- router
- wifi
- internet

are strong Technical clues.

TF-IDF is excellent when vocabulary itself provides strong category separation.

Embeddings become particularly useful when meaning needs to generalize across different wording.

For example:

- "cannot sign in"
- "unable to access my account"
- "authentication keeps failing"
- "login does not work"

may express similar meaning despite using different words.

##### 26 — Two Models Exist in This Pipeline

One conceptual point is worth making explicit.

pipeline now contains:

``` text

SentenceTransformer
        ↓
representation model

Logistic Regression
        ↓
classification model

```

More precisely:

``` text


TEXT

"My router keeps disconnecting"
        ↓
────────────────────────────────
PRETRAINED EMBEDDING MODEL
SentenceTransformer
────────────────────────────────
        ↓
384 numerical features
        ↓
────────────────────────────────
TRAINED CLASSIFIER
Logistic Regression
────────────────────────────────
        ↓
Technical

```
We trained: Logistic Regression using our support-ticket dataset.
We did not train: SentenceTransformer in this notebook.

##### Step 27 — How This Leads to Notebook 09

This notebook gives us:

``` text

Embeddings
   ↓
Logistic Regression

```

Notebook 09 will change the classifier:

``` text

Embeddings
   ↓
Neural Network

```

So our progression becomes beautifully controlled:

``` text

Experiment 1
TF-IDF
   ↓
Logistic Regression


Experiment 2
Embeddings
   ↓
Logistic Regression


Experiment 3
Embeddings
   ↓
Neural Network

```

That allows us to ask two separate questions.

Question 1

Does semantic representation help?

TF-IDF vs Embeddings
while keeping Logistic Regression


Question 2

Does nonlinear learning help?

Logistic Regression vs Neural Network
while keeping Embeddings

That's exactly why we shouldn't jump directly from TF-IDF Logistic Regression to a Transformer and change everything at once.

###### Key Learnings

The central mental model for Notebook 08 is:

``` text

TEXT
 ↓
REPRESENTATION
 ↓
NUMERIC FEATURES
 ↓
CLASSIFIER
 ↓
PREDICTION
 ↓
EVALUATION

```

For two experiments:

TF-IDF experiment

``` text

Text
 ↓
TF-IDF
 ↓
85 sparse vocabulary features
 ↓
Logistic Regression
 ↓
Category

```

versus:


Embedding experiment

``` text

Text
 ↓
SentenceTransformer
 ↓
384 dense semantic features
 ↓
Logistic Regression
 ↓
Category

```

And perhaps the most important code-level difference is:
``` text

TF-IDF
      ↓
fit_transform(train)
transform(validation)
transform(test)

versus:

Pretrained embedding model
      ↓
encode(train)
encode(validation)
encode(test)

```

because the embedding model has already been pretrained.